# Figure: Segmentation Mask Overlay

Overlays the exclusive anatomical segmentation masks (lung / soft-tissue / bone)
on sample input images from the experiment.

Mask channel order follows `MASK_GROUPS`: `['lung', 'soft', 'bone']`.

**Requires:** `nb_run.ipynb`

Output: `build/segmentation_masks.png`

In [ ]:
%%capture
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    DRIVE_ROOT = "/content/drive/MyDrive/thesis"  # <-- edit if your Drive path differs

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", f"{DRIVE_ROOT}/notebooks/requirements.txt",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "-e", DRIVE_ROOT,
    ])

    # Change to figures/ so load_results() can find the cached data
    os.chdir(f"{DRIVE_ROOT}/figures")


In [ ]:
import sys, os, glob, joblib

import matplotlib.pyplot as plt
from chest_xray_sim.visualization import plot_segmentation_overlay

def load_results(results_dir="results"):
    hits = sorted(glob.glob(f"{results_dir}/*/imgs.joblib"))
    if not hits:
        raise FileNotFoundError(f"No results in '{results_dir}'. Run nb_run.ipynb first.")
    d = os.path.dirname(hits[-1])
    print(f"Loading from {d}")
    return {
        "imgs":       joblib.load(f"{d}/imgs.joblib"),
        "masks":      joblib.load(f"{d}/masks.joblib"),
        "seg_labels": joblib.load(f"{d}/seg_labels.joblib"),
    }

In [ ]:
r = load_results()
imgs, masks, seg_labels = r["imgs"], r["masks"], r["seg_labels"]
print(f"Image shape: {imgs[0].shape}, masks: {masks[0].shape}, labels: {seg_labels}")

In [ ]:
# Single sample
IDX = 0
fig, ax = plot_segmentation_overlay(imgs[IDX], masks[IDX], seg_labels)
ax.set_title(f"Sample {IDX}: segmentation overlay")
fig.tight_layout()
fig.savefig("build/segmentation_masks.png", bbox_inches="tight")
plt.show()

In [ ]:
# Grid of 4 samples
N = min(4, imgs.shape[0])
fig, axes = plt.subplots(1, N, figsize=(5 * N, 5))
if N == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    plot_segmentation_overlay(imgs[i], masks[i], seg_labels, ax=ax)
    ax.set_title(f"Sample {i}")

fig.tight_layout()
fig.savefig("build/segmentation_masks_grid.png", bbox_inches="tight")
plt.show()